In [80]:
import pandas as pd
import re
from collections import Counter

In [81]:
file_path = 'Last_recipe.csv'
df = pd.read_csv(file_path)

# 재료명 추출을 위한 함수 정의
def extract_ingredients(recipe):
    
    recipe = recipe.replace('\u200b', '')
    recipe_without_parentheses = re.sub(r'\([^)]*\)', '', recipe)
    # 재료명에서 단위와 양 제거
    ingredients = re.sub(r'\d+([/~.+]\d+)?[a-zA-Z가-힣]*\s*|\+', '', recipe_without_parentheses)
    # 공백 제거 및 리스트 형식으로 반환
    ing = [ingredient.strip() for ingredient in ingredients.split('|')]
    return ing

remove_words = ['개', '큰술', '컵', '작은술', '꼬집', '적당량', '약간', '조각', '간',
                '줌', '스푼', '공기', '인분', '팩', '쪽', '봉지', '모', '알', 
                '통', '포기', '줄', '숟가락', '살짝', '봉', '사발', '가닥', 
                '톨', '선택', '톡톡', '톡', '조금', '넉넉하게', '넉넉히', '고운',
                '적당히', '것', '중간크기', '솔솔', '뜨거운', '따뜻한', '따듯한', '차가운', 
                '듬뿍', '취향껏', '다진', '소량', '가는', '송송', '썬', '삶은', '굵은',
                '작은거', '부분', '흰', '작은것', '불린', '간맞추기', '녹인', '작은', '미지근한',
                '대', '초록부분', '비정제', '유기농', '부순', '/', '~', '-', 'g',
                '아주조금', '채썬', '데친', '중', '잡채용', '약간씩', '자른', '큰거', '큰것',
                '흰대', '크게', '시판', '갈은', '깐', '기호에맞게', '흰부분', '채', '마리','팍팍',
                '티스푼', '우린물', '작은크기', '잘', '입맛에맞게', '종이컵', '찬', '찐', '큰',
                'T', '.', '소', '다진것', '굵은거', '기호에따라', '조금씩', '작은크기', '작은사이즈',
                '적당', '적정량', '원하는만큼', '작은스푼', '밥스푼', '반스푼', '쏭쏭', '데운', '썰은',
                '중사이즈', '저민', '기호껏', '한토막', '수저', '편', '취향것', '남은', '꼬들', '송송썬',
                'A'
                ]

def filter_ingredients(ingredient_list):
    filtered_nouns = []
    for ingredient in ingredient_list:
        words = ingredient.split()
        filtered_ingredient = ' '.join([word for word in words if word not in remove_words])
        filtered_nouns.append(filtered_ingredient)
    # 빈 문자열 제거
    filtered_nouns = [noun for noun in filtered_nouns if noun]
    return filtered_nouns

replace_dict = {'물': ['생수', '찬물', '뜨거운물', '따뜻한물', '냉수', '미지근한물', '차가운물'], 
                '소금': ['굵은소금', '꽃소금', '고운소금', '가는소금', '천일염', '절임용 소금'],
                '구운소금':['죽염', '볶음 소금', '볶은 소금'],
                '허브솔트':['허브소금','허브맛솔트','소금 후추','마늘허브소금'],
                '마늘': ['다진마늘', '간마늘', '통마늘', '편마늘', '깐마늘', '마늘다진것'],
                '양파': ['다진양파', '간양파', '양파채', '채썬양파','양파 소', '양파껍질'],
                '적양파': ['자색양파'],
                '대파': ['다진대파', '대파 흰부분', '대파흰부분', '대파잎', '대파채'],
                '대파뿌리':['대파 뿌리'],
                '고추':['고추씨', '다진고추'],
                '고춧가루': ['고추가루', '고운고춧가루', '굵은고춧가루', '고운고추가루', '굵은고추가루'],
                '청양고춧가루': ['매운고춧가루', '청양고추가루', '매운 고춧가루'],
                '고추기름': ['고춧기름', '고추씨기름'],
                '설탕': ['황설탕', '백설탕', '갈색설탕', '흰설탕', '백설스위트리자일로스설탕', 
                       '바닐라설탕', '알룰로스', '자일로스설탕', '에리스리톨', '비정제황설탕', '유기농설탕',
                       '비정제설탕', '설탕A', '설탕B', '마스코바도 설탕'],
                '깨': ['통깨', '갈은깨', '볶은참깨', '볶은 참깨', '부순깨', '볶은깨', '볶음깨', '볶은통깨', '깨가루', '깻가루'],
                '후추': ['통후추', '후추가루', '소금후추', '백후추', '흰후추', '흑후추', '후추톡톡','그라인드후추'],
                '간장': ['진간장', '양조간장'],
                '맛간장': ['만능간장', '조림간장', '대게백간장', '저염 간장', '저염간장', '다시마간장', '초간장', '양념간장', '홍게간장', '아기 간장', '어간장'],
                '국간장': ['집간장', '조선간장'],
                '발사믹식초': ['발사믹 식초', '발사믹글레이즈', '발사믹소스', '발사믹크림', '발사믹드레싱'],
                '식초': ['양조식초','현미식초','감식초','사과식초','레몬식초'],
                '올리고당': ['올리고 당', '백설올리고당'],
                '계란': ['달걀', '달걀노른자', '삶은달걀', '달걀흰자', '달걀 노른자', '달걀 흰자',
                         '달걀물', '달걀지단', '삶은계란', '계란노른자', '계란 노른자', '계란흰자',
                         '계란 흰자', '계란물', '계란지단', '노른자', '계란 계란', '실온 계란', '흰자',
                         '계란후라이', '달걀후라이', '구운 계란', '유정란'],
                '청양고추': ['다진청양고추', '청량고추', '매운고추', '청양 고추', '청양초', '땡초', 
                            '청량초','청양홍고추','홍청양고추', '매운 고추'],
                '맛술': ['맛술 미림', '요리술'], 
                '버터': ['무염버터', '녹인버터', '실온 버터', '가염버터', '실온버터', '무염 버터'],
                '밥': ['찬밥', '공기밥', '현미밥', '쌀밥', '흰밥', '잡곡밥', '즉석밥', '흰밥', '잡곡밥', '흰쌀밥', '흑미밥', '식은밥', '햇반'],
                '홍고추': ['청홍고추', '다진홍고추', '빨간고추', '마른홍고추', '붉은 고추', '레드페퍼', '붉은고추'], 
                '감자': ['알감자', '삶은감자'], 
                '무': ['무우', '무채',' 무생채'],
                '된장': ['집된장', '시판된장'],
                '파': ['다진파', '썬파', '파채'], 
                '쪽파': ['잔파', '다진쪽파'], 
                '오이': ['백오이', '오이채', '청오이'],
                '밀가루': ['통밀가루', '밀가루중력분', '중력분', '밀가루 밀가루', '우리밀'],
                '밀가루 박력분':['밀가루박력분', '밀가루 박력분', '박력분'],
                '밀가루 강력분':['밀가루강력분', '강력분', '밀가루 강력분'],
                '부추': ['영양부추'],
                '양배추': ['적양배추', '양배추채', '양배추잎'], 
                '매실액': ['매실액기스', '매실엑기스', '매실 진액', '매실진액', '매실원액'],
                '표고버섯': ['건표고버섯', '생표고버섯', '마른표고버섯'],
                '생강': ['다진생강', '간생강', '편생강'], 
                '파슬리가루': ['파슬리 가루'],
                '김치': ['김치국물', '배추김치', '김장김치', '김치 국물', '다진김치', '김칫국물', '볶은김치', '포기김치'],
                '묵은김치': ['묵은지', '익은김치', '신 김치', '익은 김치', '묵은 김치', '신김치'],
                '파프리카': ['빨간 파프리카', '노란 파프리카', '빨강 파프리카', '노랑파프리카', 
                         '빨강파프리카', '노랑 파프리카', '빨간파프리카', '노란파프리카', '홍파프리카',
                         '주황 파프리카', '미니파프리카', '주황파프리카', '파프리카 빨강', '파프리카 노랑',
                         '미니 파프리카', '초록 파프리카'],
                '피망':['청피망', '홍피망', '노랑피망'],
                '새우젓': ['새우젓국물','새우젓 국물'], 
                '올리브유':['올리브오일', '올리브 오일', '엑스트라버진 올리브유', '백설안달루시아산올리브유'],
                '액젓': ['멸치액젓', '까나리액젓', '참치액젓', '멸치 액젓', '까나리 액젓', '참치 액젓',
                       '참치액', '참치 액', '갈치액젓', '맑은 액젓'],
                '돼지고기': ['돼지고기 잡채용', '돼지고기 찌개용'],
                '돼지고기 안심': ['돼지고기안심','돼지안심'], 
                '돼지고기 등심':['돼지고기등심','돼지등심'],
                '돼지고기 다짐육': ['돼지고기다짐육', '다진돼지고기', '간돼지고기'],
                '돼지고기 앞다리살':['돼지고기앞다리살','돼지앞다리살'],
                '돼지고기 목살':['돼지고기목살', '돼지고기 돼지목살'],
                '돼지고기 삼겹살':['삼겹살','돼지고기삼겹살'],
                '소고기': ['쇠고기', '한우', '소고기 소고기'],
                '소고기 다짐육': ['소고기다짐육', '다진소고기', '다진쇠고기'],
                '소고기 국거리': ['소고기국거리', '소고기 국거리', '소고기 국거리용', '국거리용 소고기'],
                '소고기 불고기용': ['불고기용 소고기', '소고기불고기용', '소불고기', '소고기 불고기감', '불고기용소고기', '소불고기감'],
                '소고기 샤브샤브용':['소고기 샤브샤브용', '샤브샤브용 소고기'],
                '소고기 안심':['소고기안심'], '소고기 부채살':['부채살'],
                '소고기 등심':['소고기등심'],
                '소고기 사태':['소고기사태'],
                '소고기 양지':['소고기양지', '양지', '양지머리'],
                '새송이버섯': ['새송이', '새송이 버섯', '미니새송이버섯','새송이', '새송이버섯 버섯', '미니 새송이버섯'],
                '어묵': ['사각어묵', '사각 어묵', '납작어묵', '오뎅', '납작 어묵'],
                '건새우': ['마른새우', '밥새우'], '베이킹파우더':['베이킹 파우더', '베이킹파우다', 'bp', 'b.p'], '베이킹소다':['베이킹 소다'],
                '새우': ['칵테일새우', '냉동새우', '새우살', '생새우', '칵테일 새우', '보리새우', '자숙새우', '냉동 새우', '흰다리새우'],
                '멸치':['잔멸치', '다시멸치', '국물용 멸치', '중멸치', '국물용멸치','다시 멸치', '디포리'],
                '멸치육수': ['멸치 육수', '국멸치', '국물멸치', '멸치다시물', '멸치다시팩', '멸치다시육수', '멸치다싯물',
                            '다시육수', '다시팩', '다시물'],
                '다시마': ['다시마육수', '다시마물', '건다시마', '다시마 육수', '다시마우린물', '다시마 우린 물', '다시마 물', '다시마가루'],
                '멸치다시마': ['멸치다시마육수', '멸치 다시마 육수', '다시마 멸치육수', 
                              '다시마멸치육수', '멸치다시마 육수', '멸치육수팩'],
                '방울토마토': ['방울 토마토'], 
                '시금치':['시금치나물'], 
                '들깨가루':['들깻가루'],
                '느타리버섯': ['애느타리버섯', '느타리 버섯', '느타리'],
                '모짜렐라치즈':['모차렐라치즈', '모짜렐라 치즈', '모차렐라치즈', '모차렐라 치즈', '생모짜렐라치즈', '모짜렐라', '피자 치즈'],
                '고구마': ['호박고구마', '찐고구마', '밤고구마', '삶은고구마', '자색고구마', '찐 고구마'],
                '햄': ['슬라이스햄', '슬라이스 햄', '통조림햄', '샌드위치햄', '리챔', '캔햄'],
                '닭가슴살': ['닭 가슴살', '훈제닭가슴살', '닭가슴살캔'], 
                '월계수잎': ['월계수', '월계수잎 잎'],
                '케찹': ['토마토케찹', '케첩', '토마토케첩', '케챱', '케챂', '캐첩', '토마토 케찹', '케쳡'],
                '쌀': ['쌀뜨물', '쌀가루', '불린쌀', '쌀뜬물', '멥쌀', '멥쌀가루', '백미', '맵쌀', '흑미'],
                '김': ['김가루', '조미김', '구운김'], 
                '미나리':['돌미나리'],
                '김밥김': ['김밥용김', '김밥용 김', '김밥 김'],
                '당면': ['불린당면'], 
                '브로콜리':['브로컬리', '데친브로콜리'], 
                '상추':['청상추'],
                '숙주': ['숙주나물'], 
                '기름': ['튀김유', '튀김기름', '오일', '부침유', '백설요리유', '요리유', '튀김용기름', '식물성오일', '식물성 기름', ],
                '양송이버섯': ['양송이', '양송이 버섯', '양송이버섯 버섯'],
                '전분가루': ['전분', '전분물', '감자전분', '옥수수전분', '옥수수 전분','감자전분가루', '감자 전분가루', '감자 전분'],
                '맛살':['게맛살', '크래미', '게살'], 
                '얼음':['얼음물', '각얼음'],
                '참치캔': ['캔참치', '참치통조림', '고추참치', '참치 캔', '참치 통조림'],
                '풋고추': ['청고추', '아삭이고추', '오이고추'],
                '건고추': ['마른고추'], 
                '대패삼겹살': ['대패 삼겹살'], 
                '단호박': ['미니단호박', '미니 단호박'],
                '토마토소스': ['토마토 소스', '토마토페이스트', '토마토스파게티소스','토마토 페이스트', '토마토파스타소스', '토마토 스파게티소스'],
                '딸기잼': ['딸기쨈'], 
                '알배추':['알배기배추', '알배기 배추'], 
                '굴':['생굴'], 
                '굴소스':['굴 소스'],
                '바지락': ['바지락살'], 
                '찹쌀':['찹쌀풀', '찹쌀가루', '불린찹쌀'], 
                '라면': ['라면사리','신라면','라면스프'],
                '파마산치즈가루': ['파마산치즈', '파마산 치즈가루', '파마산 치즈', '파마산치즈가루 가루',
                                 '파르메산치즈가루','파마산가루'],
                '조청': ['쌀조청'], 
                '미역': ['건미역', '미역줄기', '불린미역', '마른미역', '자른미역', '마른 미역', '염장미역줄기'],
                '스파게티면': ['스파게티 면', '스파게티', '파스타면', '파스타', '스파게티면 면'], 
                '소세지': ['소시지', '분홍소세지', '분홍소시지'],
                '후랑크소세지': ['프랑크소시지', '프랑크소세지', '후랑크소시지', '프랑크 소세지', '후랑크 소세지'],
                '비엔나소세지': ['비엔나소시지', '비엔나 소세지', '비엔나 소시지', '비엔나'],
                '닭': ['닭고기', '닭육수', '생닭',' 닭볶음탕용 닭', '토종닭', '닭 볶음탕용', '닭볶음탕용 닭'], 
                '닭봉':['닭날개'],
                '닭다리살': ['닭다리'], '닭안심': ['닭안심살', '닭 안심'], '다시다': ['소고기다시다'],
                '아몬드가루': ['아몬드 가루', '아몬드파우더', '아몬드분말'], 
                '아몬드슬라이스': ['슬라이스아몬드', '슬라이스 아몬드', '아몬드 슬라이스'],
                '홍합': ['홍합살'], 
                '와사비': ['생와사비'], 
                '슬라이스치즈': ['슬라이스 치즈', '체다슬라이스치즈', '슬라이스 체다치즈', '체다 슬라이스치즈', '슬라이스체다치즈','체다 치즈', '체더 치즈', '아기 치즈'],
                '옥수수콘': ['캔옥수수', '콘옥수수', '옥수수캔', '옥수수통조림', '스위트콘', '옥수수 통조림', '캔 옥수수','통조림 옥수수', '통조림옥수수'],
                '코코아가루': ['코코아파우더', '코코아 파우더', '무가당 코코아가루', '코코아 가루'], 
                '어린잎채소': ['어린잎', '베이비채소', '새싹채소', '새싹', '어린잎채소 채소'],
                '칠리소스': ['스위트칠리소스'], 
                '우렁이':['우렁'],
                '이스트': ['드라이이스트', '인스턴트드라이이스트', '인스턴트 드라이 이스트', '인스턴트 드라이이스트','드라이 이스트', '인스턴드라이이스트'],
                '머스타드': ['머스터드', '허니머스타드', '머스터드소스', '머스타드소스', '홀그레인머스타드', '허니머스타드소스',
                         '허니머스터드', '홀그레인머스터드', '허니머스터드소스', '홀그레인 머스타드', '머스타드 소스', '허니 머스타드', '씨겨자'],
                '메이플시럽': ['메이플 시럽'], '계피가루':['시나몬가루', '시나몬 가루', '시나몬파우더', '계핏가루', '시나몬 파우더'],
                '피클': ['다진피클', '오이피클', '오이지'],
                '떡볶이떡':['떡볶이 떡', '밀떡', '쌀떡'],
                '식용유': ['식용류'],
                '요거트':['플레인요거트', '플레인 요거트', '그릭요거트','요플레','무가당 요거트'],
                '칼국수면': ['칼국수', '칼국수면 면'],
                '돼지등갈비':['등갈비'],
                '페페론치노': ['페퍼론치노', '페페로치노'],
                '우동면':['우동사리', '우동'],
                '우스터소스':['우스타소스'],
                '카레가루': ['카레', '고형카레', '카레가루 가루'],
                '바닐라익스트랙':['바닐라 익스트랙', '바닐라에센스', '바닐라액', '바닐라 에센스', '바닐라 기름', '바닐라익스트렉', '바닐라 익스트렉'],
                '팥':['팥앙금', '팥배기'],
                '커피':['커피가루', '인스턴트커피', '인스턴트 커피'],
                '명란':['명란젓', '저염 명란'],
                '다크초콜릿':['초콜릿', '다크 다크초콜릿'],
                '천연조미료':['천연조미료육수'],
                '호박': ['둥근호박', '늙은호박'],
                '크랜베리': ['건크랜베리'],
                '슈가파우더':['슈거파우더', '슈가 파우더'],
                '돈까스소스': ['돈가스소스', '돈까스 소스'],
                '마늘쫑': ['마늘종'],
                '채소':['야채','각종야채', '냉장고속야채', '채소육수', '각종채소', '채수'],
                '당근':['당근채', '다진당근',' 당근라페'],
                '얼갈이배추': ['얼갈이', '단배추'], '바게트':['바게트빵'],
                '피쉬소스': ['피시소스'],
                '쭈꾸미':['주꾸미'],
                '녹말가루': ['녹말', '녹말물', '녹말', '물녹말'],
                '분유':['탈지분유'],
                '젤라틴':['판젤라틴'],
                '초고추장': ['초장', '초곡추장'],
                '코인육수':['코인 육수'], 
                '표고버섯':['건표고버섯', '말린표고버섯','건표고','말린 표고버섯', '생표고버섯',
                           '표고', '마른표고버섯', '말린표고버섯', '마른표고', '마른 표고버섯',
                           '표고버섯기둥'],
                '표고버섯가루':['표고가루', '표고버섯 가루'],
                '플레인요구르트': ['플레인 요구르트'],
                '떡국떡':['떡국 떡'],
                '북어채': ['북어포', '북어'],
                '딸기잼':['잼'],
                '스테이크소스':['스테이크 소스'],
                '핫케이크가루': ['핫케익가루', '핫케익믹스', '핫케이크 가루', '핫케익가루'],
                '돈까스':['돈가스'],
                '사골육수':['사골곰탕', '사골국물'],
                '오징어':['오징어 몸통', '오징어몸통', '오징어다리', '물오징어'],
                '샐러리':['셀러리'],
                '올리브':['블랙 올리브', '그린올리브', '블랙올리브'], 
                '땅콩버터':['피넛버터','땅콩잼', '땅콩소스'],
                '샐러드채소':['샐러드야채', '샐러드', '샐러드채소 채소'],
                '골뱅이':['골뱅이통조림', '골뱅이국물', '골뱅이캔', '골뱅이 국물'],
                '포도씨유':['포도씨오일'],
                '그린빈':['그린빈스'],
                '주키니호박':['쥬키니호박'],
                '메추리알':['깐메추리알'],
                '가다랑어포':['가스오부시', '가쓰오부시',],
                '김밥용 햄':['김밥용햄', '김밥햄'],
                '김밥용 단무지':['김밥용단무지'],
                '대추':['건대추'],
                '유부':['유부초밥'],
                '키위':['골드키위'],
                '화이트와인':['화이트 와인'],
                '해물믹스':['해물', '모둠해물', '해물믹스', '냉동해물','모듬해물'],
                '해물육수':['천연조미료해물육수'],
                '훈제오리':['훈제오리고기', '오리훈제','오리'],
                '휘핑크림':['동물성 휘핑크림'],
                '고사리':['고사리나물'],
                '딸기잼':['딸기쨈'],
                '치킨':['남은치킨'],
                '쌈무':['무쌈'],
                '전복':['전복내장'],
                '무청':['무청시래기', '삶은시래기'],
                '호두':['다진호두', '호두분태'],
                '토마토':['완숙토마토','완숙 토마토'],
                '마스카포네치즈':['마스파코네 치즈'],
                '블루베리':['냉동블루베리', '냉동 블루베리', '건블루베리'],
                '바닐라아이스크림':['바닐라 아이스크림'],
                '파인애플':['파인애플통조림'],
                '문어':['자숙문어'],
                '연두':['오리에센스 연두'],
                '양상추':['양상치'],
                '민트':['민트잎'],
                '레몬':['레몬껍질'],
                '레몬주스':['레몬쥬스'],
                '김':['구운 김'],
                '두부':['부침용 두부'],
                '아보카도오일':['아보카도 기름'],
                '파프리카가루':['파프리카 가루', '파프리카파우더', '훈제파프리카가루'],
                '후리가케':['후리카케'],
                '견과류':['하루견과'],
                '옥수수':['찰옥수수'],
                '짜장가루':['짜장분말'],
                '열무김치':['열무김치국물'],
                '연어':['훈제연어', '생연어', '연어캔','연어통조림'],
                '꿀':['벌꿀'],
                '고추장':['집고추장', '약고추장'],
                '바질':['건바질'],
                '꼬막':['꼬막살'],
                '노두유':['노추'],
                '코코넛오일':['뷰코 엑스트라버진 코코넛 기름'],
                '코코넛밀크':['뷰코 코코넛 밀크 프리미엄'],
                '아기 간장':['아기간장'],
                '배추':['절임배추', '배춧잎', '배추잎'],
                '우유':['흰우유'],
                '마스카포네치즈':['마스카포네 치즈'],
                '만두':['군만두','고기만두', '냉동만두'],
                '튀김가루':['치킨튀김가루'],
                '고등어':['노르웨이 고등어', '고등어통조림'],
                '시리얼':['씨리얼'],
                '겨자':['연겨자', '겨자소스'],
                '와사비':['연와사비'],
                '연두':['요리에센스 연두'],
                '검은깨':['검정깨'], '검은콩':['검정콩'],
                '치킨스톡':['치킨파우더'],
                '스파게티소스':['스파게티면 소스'],
                '냉면육수':['냉면 육수'],
                '짜장라면':['짜파게티'],
                '밤':['생밤', '알밤', '깐밤'],
                '크랜베리':['크렌베리', '크린베리'],
                '베이크드빈스':['베이크드빈'],
                '베트남고추':['베트남 고추'],
                '오리엔탈드레싱':['오리엔탈소스']
                }

# 역방향 사전 생성
reverse_dict = {value: key for key, values in replace_dict.items() for value in values}
sorted_keys = sorted(reverse_dict.keys(), key=len, reverse=True)

# 정규식 패턴 생성
pattern = re.compile(r'\b(' + '|'.join(re.escape(key) for key in sorted_keys) + r')\b')

def unify_ingredients(ingredient_list):
    unified = []
    for ingredient in ingredient_list:
        # 정규식 대체 함수 정의
        def replace_func(match):
            return reverse_dict[match.group(0)]
        # 정규식을 한 번만 적용하여 대체
        unified_ingredient = pattern.sub(replace_func, ingredient, count=1)
        
        words = unified_ingredient.split()
        seen = set()
        result = []
        for word in words:
            if word not in seen:
                seen.add(word)
                result.append(word)

        unified.append(' '.join(result))
    return unified
            

# 'recipeIngredient' 열에서 재료명 추출
df['recipeIngredient1'] = df['recipeIngredient'].apply(extract_ingredients)
# 'remove_words'를 제외한 단어만 추출
df['recipeIngredient2'] = df['recipeIngredient1'].apply(filter_ingredients)
# 'replace_dict'를 통해 동일한 재료명 병합
df['recipeIngredient3'] = df['recipeIngredient2'].apply(unify_ingredients)
df['recipeIngredient3'] = df['recipeIngredient3'].apply(unify_ingredients)

# 재료명을 키로, 해당 재료가 포함된 레시피 인덱스를 값으로 가지는 딕셔너리 생성

ingredients_dict = {}

for idx, ingredients_list in df['recipeIngredient3'].items():
    for ingredient in ingredients_list:
        if ingredient in ingredients_dict:
            ingredients_dict[ingredient] += 1
        else:
            ingredients_dict[ingredient] = 1
            
sorted_ingredients_dict = dict(sorted(ingredients_dict.items(), key=lambda item: item[1], reverse=True))

filtered_sorted_ingredients_count_dict = {k: v for k, v in sorted_ingredients_dict.items() if v >= 50}

with open('sorted_ingredients_count.txt', 'w') as f:
    for ingredient, count in filtered_sorted_ingredients_count_dict.items():
        f.write(f'{ingredient}:{count}\n')

In [82]:
# txt 파일에서 재료명 읽어오기
with open('sorted_ingredients_count_updated.txt', 'r') as file:
    ingredients_list = [line.split('|')[0] for line in file]

# 각 레시피가 txt 파일의 재료명을 포함하는지 확인
def contains_ingredients(ingredient_list, valid_ingredients):
    return all(ingredient in valid_ingredients for ingredient in ingredient_list) 

# 유효한 재료명 리스트
valid_ingredients = set(ingredients_list)

# 유효한 재료명을 포함하는 레시피만 남기기
df_filtered = df[df['recipeIngredient3'].apply(lambda x: contains_ingredients(x, valid_ingredients))]

In [83]:
# 각 메뉴 이름별로 재료 리스트 병합
df_filtered = df_filtered.groupby('best_name').filter(lambda x: len(x) > 1)
grouped_ingredients = df_filtered.groupby('best_name', group_keys=False)['recipeIngredient3'].sum()

# 각 메뉴 이름에 대해 가장 많이 사용된 3개의 재료 추출하는 함수 -> 핵심재료
def get_top_3_ingredients(ingredients_list):
    counter = Counter(ingredients_list)
    top_3 = counter.most_common(3)
    return [ingredient for ingredient, count in top_3]

# 각 메뉴 이름에 대해 top 3 재료 추출
df_filtered['top_3_per_menu'] = df_filtered['best_name'].map(lambda name: get_top_3_ingredients(grouped_ingredients[name]))

In [84]:
df_filtered = df_filtered.drop(columns=['author', 'datePublished', 'recipeIngredient',
                                        'Ingr1', 'Ingr2', 'recipeIngredient1', 'recipeIngredient2',
                                        'rating', 'rating_count'])
print(df_filtered.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 90459 entries, 2 to 198239
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   index              90459 non-null  int64 
 1   name               90459 non-null  object
 2   best_name          90459 non-null  object
 3   totalTime          90459 non-null  object
 4   recipeYield        90459 non-null  object
 5   recipeLev          90459 non-null  object
 6   description        90459 non-null  object
 7   by_sort            90022 non-null  object
 8   by_ingredient      90022 non-null  object
 9   by_situation       87827 non-null  object
 10  image1             90459 non-null  object
 11  image2             90459 non-null  object
 12  recipeIngredient3  90459 non-null  object
 13  top_3_per_menu     90459 non-null  object
dtypes: int64(1), object(13)
memory usage: 10.4+ MB
None


In [85]:
df_filtered.head()

,index,name,best_name,totalTime,recipeYield,recipeLev,description,by_sort,by_ingredient,by_situation,image1,image2,recipeIngredient3,top_3_per_menu
2,6992711,두부 구이 어묵 볶음 밥반찬 도시락 간단레시피,두부구이,PT15M,5 servings,초급,바쁠때는 모두 볶아주세요,밑반찬,콩/견과류,일상,https://recipe1.ezmember.co.kr/cache/recipe/20...,https://recipe1.ezmember.co.kr/cache/recipe/20...,"[두부, 어묵, 대파, 멸치, 식용유, 설탕, 간장, 깨, 소금]","[두부, 간장, 마늘]"
4,6962203,[무알콜모히또 만드는 법]상큼한 라임에이드 논알콜 과일 음료,모히또,PT10M,2 servings,아무나,"상큼한 라임에이드, 논알콜 과일 음료, 무알콜모히또~!!",차/음료/술,과일류,일상,https://recipe1.ezmember.co.kr/cache/recipe/20...,https://recipe1.ezmember.co.kr/cache/recipe/20...,"[라임, 애플민트, 탄산수, 얼음, 설탕]","[라임, 탄산수, 얼음]"
5,6836645,건새우 고추장 볶음,건새우고추장볶음,PT10M,6 servings,초급,건새우는 된장 찌개나 수제비등에 들어가 국물로 우려 먹게 되지만 이렇게 가끔은 고추...,밑반찬,건어물류,일상,https://recipe1.ezmember.co.kr/cache/recipe/20...,https://recipe1.ezmember.co.kr/cache/recipe/20...,"[건새우, 식용유, 고추장, 마늘, 참기름, 물엿, 설탕, 깨]","[건새우, 고추장, 마늘]"
8,6999802,쭈꾸미볶음 레시피 간단한 양념장,주꾸미볶음,PT15M,3 servings,아무나,이름만 들어도 맛있는 쭈꾸미볶음 레시피를 준비했다. 물기가 살짝 있는 버전이다. 집...,밥/죽/떡,해물류,일상,https://recipe1.ezmember.co.kr/cache/recipe/20...,https://recipe1.ezmember.co.kr/cache/recipe/20...,"[쭈꾸미, 대파, 양파, 양배추, 찹쌀, 물, 청양고추, 기름, 고춧가루, 마늘, ...","[쭈꾸미, 마늘, 고춧가루]"
10,6914579,삼삼칩스&인삼견과딥소스,칩,PT30M,1 servings,아무나,우리의 인삼을 넣어 고소하고 쌉싸름하게 즐기는 견과딥소스에 바삭하게 튀긴 인삼칩을 ...,과자,기타,일상,https://recipe1.ezmember.co.kr/cache/recipe/20...,https://recipe1.ezmember.co.kr/cache/recipe/20...,"[인삼, 튀김가루, 전분가루, 기름, 소금, 수삼, 아몬드, 잣, 두유, 포도씨유,...","[소금, 인삼, 튀김가루]"


In [86]:
input_file = 'sorted_ingredients_count_updated.txt'
output_file = 'last_ingredients.txt'

# 파일 읽기 및 처리
with open(input_file, 'r') as file:
    lines = file.readlines()

# 새로운 라인 생성
processed_lines = []
for idx, line in enumerate(lines):
    parts = line.strip().split('|')
    if len(parts) == 3:
        ingredient = parts[0]
        category = parts[1]
        # 새로운 포맷: 번호|재료명|카테고리
        new_line = f"{idx + 1}|{ingredient}|{category}\n"
        processed_lines.append(new_line)

# 새로운 파일에 저장
with open(output_file, 'w') as file:
    file.writelines(processed_lines)


In [87]:
input_file = 'last_ingredients.txt'

data = []
with open(input_file, 'r') as file:
    for line in file:
        parts = line.strip().split('|')
        if len(parts) == 3:
            num = int(parts[0])
            ingredient = parts[1]
            category = parts[2]
            data.append([num, ingredient, category])
            
df = pd.DataFrame(data, columns= ['Index', 'Ingredient', 'Category'])

In [102]:
df.to_csv('last_ingredients.csv', index=False, encoding='utf-8-sig')

In [88]:
# 매핑 함수 정의
def map_ingredients_to_numbers(ingredients, mapping_dict):
    return [mapping_dict[ingredient] for ingredient in ingredients]

# 'recipeIngredient3' 열에 매핑 함수 적용
df_filtered['recipeIngredient3'] = df_filtered['recipeIngredient3'].apply(lambda x: map_ingredients_to_numbers(x, mapping_dict))
df_filtered['top_3_per_menu'] = df_filtered['top_3_per_menu'].apply(lambda x: map_ingredients_to_numbers(x, mapping_dict))


print(df_filtered)


          index                                name best_name totalTime  \
2       6992711          두부 구이 어묵 볶음 밥반찬 도시락 간단레시피       두부구이     PT15M   
4       6962203   [무알콜모히또 만드는 법]상큼한 라임에이드 논알콜 과일 음료       모히또     PT10M   
5       6836645                          건새우 고추장 볶음  건새우고추장볶음     PT10M   
8       6999802                   쭈꾸미볶음 레시피 간단한 양념장     주꾸미볶음     PT15M   
10      6914579                        삼삼칩스&인삼견과딥소스         칩     PT30M   
...         ...                                 ...       ...       ...   
198232  7005169                 생깻잎으로 깻잎김치 맛있게 만들기       깻잎김치     PT15M   
198233  6915445                     ♥[신혼밥상] 뚝딱 두부김치      두부김치     PT20M   
198234  7011128                     바지락 찜 (성시경 레시피)      바지락찜     PT15M   
198237  6896597             깻잎부추전 ☆ 막걸리생각이 간절....ㅜㅜ     깻잎부추전     PT30M   
198239  6971861  가지덮밥, 말캉한 가지 튀김이 매력적인 별미요리, 가지튀김덮밥    가지튀김덮밥   PT0H60M   

       recipeYield recipeLev  \
2       5 servings        초급   
4       2 servings       아무나   
5  

In [89]:
def extract_number(yield_str):
    match = re.search(r'\d+', yield_str)  # \d+는 하나 이상의 숫자를 의미
    if match:
        return int(match.group())
    return None

# 'recipeYield' 열에서 숫자 추출
df_filtered['recipeYield'] = df_filtered['recipeYield'].apply(extract_number)

In [90]:
def convert_to_minutes(duration_str):
    # 정규 표현식을 사용하여 시간과 분 추출
    hours = re.search(r'(\d+)H', duration_str)
    minutes = re.search(r'(\d+)M', duration_str)
    
    # 시간과 분의 기본값 설정
    total_minutes = 0
    
    if hours:
        total_minutes += int(hours.group(1)) * 60
    if minutes:
        total_minutes += int(minutes.group(1))
    
    return total_minutes

# 'duration' 열에서 총 분으로 변환
df_filtered['totalTime'] = df_filtered['totalTime'].apply(convert_to_minutes)
df_filtered.loc[df_filtered['totalTime'] == 3510, 'totalTime'] = 90

In [95]:
df_filtered.head(1)

,index,name,best_name,totalTime,recipeYield,recipeLev,description,by_sort,by_ingredient,by_situation,image1,image2,recipeIngredient3,top_3_per_menu
2,6992711,두부 구이 어묵 볶음 밥반찬 도시락 간단레시피,두부구이,15,5,초급,바쁠때는 모두 볶아주세요,밑반찬,콩/견과류,일상,https://recipe1.ezmember.co.kr/cache/recipe/20...,https://recipe1.ezmember.co.kr/cache/recipe/20...,"[32, 61, 9, 58, 15, 3, 5, 12, 2]","[32, 5, 1]"


In [97]:
df_filtered.rename(columns={'index':'id', 'best_name': 'bestName', 'totalTime': 'cookingTime',
                            'recipeLev':'difficulty', 'recipeYield':'servings',
                            'recipeIngredient3': 'RecipeIngredient'}, inplace=True)

In [98]:
x = df_filtered.head(10)

In [101]:
x.to_csv('temp.csv', index=False, encoding='utf-8-sig')

In [103]:
df_filtered.to_csv('recipe.csv', index=False, encoding='utf-8-sig')